# Allen Brain Atlas demo

Run `volumetric-qc` on a publicly downloadable Allen Brain Atlas reference volume. We use the **CCFv3 average template** (NIfTI, ~80 MB, single channel) because it requires no auth and is permissively licensed.

**Reference.** Allen Mouse Common Coordinate Framework v3 — Wang et al., *Cell* 2020. https://help.brain-map.org/display/mouseconnectivity/API

If you have a real cleared-tissue volume in OME-Zarr or OME-TIFF, the same code path runs on it unchanged — just point `open_volume` at the path.

In [ ]:
import urllib.request
from pathlib import Path

# Allen Mouse CCFv3 average template, 25 µm isotropic, NIfTI.
DATA_URL = 'https://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/average_template/average_template_25.nrrd'
LOCAL = Path('average_template_25.nrrd')

if not LOCAL.exists():
    print(f'Downloading {DATA_URL} ...')
    urllib.request.urlretrieve(DATA_URL, LOCAL)
print(f'Local file: {LOCAL.resolve()}  size: {LOCAL.stat().st_size / 1e6:.1f} MB')

## Convert NRRD to a numpy array

`volumetric-qc.io` natively reads NIfTI. The Allen template is distributed as NRRD; we convert it to an in-memory array with `pynrrd` (or `nibabel` if you re-export as NIfTI first).

In [ ]:
try:
    import nrrd
    arr, header = nrrd.read(str(LOCAL))
except ImportError:
    print('Install pynrrd to load the .nrrd file:  pip install pynrrd')
    raise

import numpy as np
arr = arr.astype(np.float32)
print('shape (xyz):', arr.shape)
# Allen NRRD axes: (x, y, z). Transpose to (z, y, x) for volumetric-qc.
vol_zyx = arr.transpose(2, 1, 0)
print('shape (zyx):', vol_zyx.shape)

## Open as a LazyVolume and run the full QC pipeline

In [ ]:
from volumetric_qc import open_volume, run_qc, QCConfig
from volumetric_qc.reports import write_html_report, write_json_summary

lv = open_volume(vol_zyx)
print('LazyVolume shape:', lv.shape)

cfg = QCConfig()
cfg.channels = ['template_avg']
cfg.voxel_size_um = (25.0, 25.0, 25.0)
cfg.sampling.z_stride = 2
cfg.sampling.xy_downsample = 2
cfg.sampling.blob_z_sample = 12
# The Allen reference is single-channel — disable cross-channel checks.
cfg.metrics.channel_bleed = False
cfg.metrics.registration = False

result = run_qc(lv, cfg, progress=True)
print(f'\noverall_pass: {result.overall_pass}')
print(f'n_fail: {result.n_fail}  n_warn: {result.n_warn}  elapsed: {result.elapsed_seconds:.1f}s')

In [ ]:
for f in result.flags:
    if f.severity != 'pass':
        print(f'{f.severity:5s} {f.name:40s} value={f.value:.4f}')

## Write the dashboard

The reference template is a smoothed group average so a few sharp-image checks will warn (the image is genuinely smooth). That's expected.

In [ ]:
html = write_html_report(result, 'allen_brain_qc.html', title='Allen CCFv3 — average template QC')
summary = write_json_summary(result, 'allen_brain_qc.json')
print('wrote', html, summary)

## Visualizing a few slices for reference

In [ ]:
import matplotlib.pyplot as plt
z = vol_zyx.shape[0]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, k in zip(axes, [z // 4, z // 2, 3 * z // 4]):
    ax.imshow(vol_zyx[k], cmap='gray')
    ax.set_title(f'z={k}'); ax.axis('off')
fig.suptitle('Allen CCFv3 — coronal slices'); plt.tight_layout(); plt.show()